In [0]:
import numpy as np
import pandas as pd
from datetime import datetime
from dateutil.relativedelta import relativedelta

ohlcv = spark.table("workspace.bronze.ohlcv_adj_close").toPandas().set_index("index")
get_nav = lambda x:(1 + x).cumprod()
get_mdd = lambda x:(x / x.cummax() - 1).cummin().iloc[-1]
today = datetime.today()
begining_of_the_year    = pd.to_datetime(datetime(today.year, 1, 1))
begining_of_the_quarter = pd.to_datetime(datetime(today.year, 3*((today.month-1)//3)+1, 1))
begining_of_the_month   = pd.to_datetime(datetime(today.year, today.month, 1))

def get_slice(x):
    return {"3y" :x[today - relativedelta(years=5) <= x.index].copy(),
            "1y" :x[today - relativedelta(years=1) <= x.index].copy(),
            "ytd":x[begining_of_the_year <= x.index].copy(),
            "qtd":x[begining_of_the_quarter <= x.index].copy(),
            "mtd":x[begining_of_the_month <= x.index].copy(),}

def capm(returns):
    nav = get_nav(returns)
    benchmark = nav["VT"]
    days = nav["VT"].shape[0]
    expected_return = nav.iloc[-1] ** (252 / days) - 1
    std_risk = 252 ** 0.5 * returns.std()
    sharpe_ratio = expected_return / std_risk
    beta = returns.cov()["VT"] / returns["VT"].var()
    alpha = expected_return - beta * expected_return["VT"]
    mdd = get_mdd(nav)
    var = returns.quantile(0.01)
    skewness = returns.skew()
    stats = pd.concat([expected_return, std_risk, sharpe_ratio, 
                       alpha, beta, mdd, var, skewness], axis=1)
    stats.columns = ["μ", "σ", "s", "α", "β", "MDD", "VaR", "γ₁"]
    stats = stats.sort_values("s", key=abs, ascending=False)
    return stats

closes = ohlcv.dropna(thresh=50)
returns = closes.pct_change(fill_method=None).iloc[1:]
slices = get_slice(returns)
res = capm(returns)

In [0]:
res

In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.gold")
spark.createDataFrame(res.reset_index()) \
        .write \
        .format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable("workspace.gold.capm") 